# 🚦 Thử nghiệm Query Router (Phân loại câu hỏi)
**Mục tiêu:** Kiểm nghiệm module `classify_query` từ `traffic_prompts.yaml`.

| Nhãn | Ý nghĩa |
|------|--------|
| **0** | Chào hỏi / Hỏi về chatbot |
| **1** | Câu hỏi liên quan Luật Giao thông ✅ |
| **2** | Câu hỏi không liên quan / thô tục |

Ta sẽ thống kê Accuracy và các trường hợp dễ nhầm.

In [ ]:
import os, sys, yaml
from dotenv import load_dotenv

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

import google.generativeai as genai
from source.core.config import Settings

settings = Settings()
api_key = settings.api_key or os.getenv('API_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.0-flash')

# Load prompts từ YAML
PROMPT_PATH = os.path.join(PROJECT_ROOT, 'source', 'core', 'traffic_prompts.yaml')
with open(PROMPT_PATH, 'r', encoding='utf-8') as f:
    prompts = yaml.safe_load(f)['prompts']

print("✅ Đã tải prompts từ traffic_prompts.yaml")
print(f"✅ Các prompts có sẵn: {list(prompts.keys())}")

## 1. Hàm gọi Router

In [ ]:
def call_router(query: str) -> str:
    """Gọi prompt classify_query, trả về '0', '1', hoặc '2'."""
    full_prompt = ""
    for msg in prompts['classify_query']['messages']:
        content = msg['content'].format(query=query)
        full_prompt += f"{'SYSTEM' if msg['role'] == 'system' else 'USER'}:\n{content}\n\n"
    response = model.generate_content(full_prompt)
    return response.text.strip()

# Test nhanh
print("🧪 Test nhanh router:")
print(f"  'xin chào bạn' → {call_router('xin chào bạn')} (kỳ vọng: 0)")
print(f"  'phạt nồng độ cồn bao nhiêu?' → {call_router('phạt nồng độ cồn bao nhiêu?')} (kỳ vọng: 1)")
print(f"  'hôm nay trời đẹp không?' → {call_router('hôm nay trời đẹp không?')} (kỳ vọng: 2)")

## 2. Dataset đánh giá Router

In [ ]:
ROUTER_TEST = [
    # Label 0 - Hỏi về chatbot
    {"query": "bạn là ai?", "expected": "0"},
    {"query": "ai tạo ra bạn?", "expected": "0"},
    {"query": "xin chào", "expected": "0"},
    {"query": "bạn có thể làm gì?", "expected": "0"},
    # Label 1 - Luật giao thông
    {"query": "xe máy uống rượu bị phạt bao nhiêu?", "expected": "1"},
    {"query": "vượt đèn đỏ ô tô 4 chỗ phạt thế nào?", "expected": "1"},
    {"query": "không đội mũ bảo hiểm bị xử phạt gì?", "expected": "1"},
    {"query": "tốc độ tối đa trên đường cao tốc là bao nhiêu?", "expected": "1"},
    {"query": "giấy phép lái xe hạng B có lái được xe 7 chỗ không?", "expected": "1"},
    {"query": "xe ô tô điện có phải đăng kiểm không?", "expected": "1"},
    # Label 2 - Không liên quan
    {"query": "hôm nay thời tiết thế nào?", "expected": "2"},
    {"query": "giá vàng hôm nay là bao nhiêu?", "expected": "2"},
    {"query": "tính toán 5 * 7 bằng mấy?", "expected": "2"},
    {"query": "mua iphone 15 ở đâu rẻ nhất?", "expected": "2"},
]

print(f"✅ Dataset đánh giá: {len(ROUTER_TEST)} câu hỏi")
label_counts = {}
for item in ROUTER_TEST:
    label_counts[item['expected']] = label_counts.get(item['expected'], 0) + 1
print(f"   Phân phối: {label_counts}")

## 3. Chạy đánh giá

In [ ]:
print("🚀 Đang chạy Router trên toàn bộ dataset...\n")
results = []
for i, item in enumerate(ROUTER_TEST):
    predicted = call_router(item['query'])
    # Lấy số đầu tiên trong response
    predicted_label = '?' 
    for c in predicted:
        if c in ['0', '1', '2']:
            predicted_label = c
            break
    
    is_correct = predicted_label == item['expected']
    results.append({
        "query": item['query'],
        "expected": item['expected'],
        "predicted": predicted_label,
        "correct": is_correct
    })
    status = "✅" if is_correct else "❌"
    print(f"  [{i+1:02d}] {status} Kỳ vọng={item['expected']} | Dự đoán={predicted_label} | '{item['query'][:50]}'")

# Tổng kết
accuracy = sum(r['correct'] for r in results) / len(results)
wrong = [r for r in results if not r['correct']]
print(f"\n{'='*60}")
print(f"📊 ACCURACY: {accuracy:.1%} ({sum(r['correct'] for r in results)}/{len(results)})")
if wrong:
    print(f"\n❌ Các câu hỏi phân loại sai ({len(wrong)}):")
    for w in wrong:
        print(f"   '{w['query']}' → Kỳ vọng: {w['expected']}, Thực tế: {w['predicted']}")

## 4. Thử nghiệm câu hỏi biên (edge cases)

In [ ]:
EDGE_CASES = [
    "đi xe đạp có bị thổi nồng độ cồn không?",
    "trẻ em được ngồi ở ghế phụ xe máy không?",
    "điều khiển xe khi mất ngủ có bị phạt không?",
    "ô tô cứu thương được phép vi phạm tốc độ không?",
]
print("🔍 Edge Cases (câu hỏi biên):")
for q in EDGE_CASES:
    label = call_router(q)
    label_clean = next((c for c in label if c in ['0','1','2']), '?')
    label_names = {'0': 'Hỏi chatbot', '1': 'Luật Giao thông', '2': 'Không liên quan', '?': 'Không rõ'}
    print(f"  Q: '{q}'")
    print(f"     → [{label_clean}] {label_names[label_clean]}\n")